In [17]:
import joblib
import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


df=pd.read_csv("clean_data.csv")

X = df['Narrative Text']
y = df['Event Label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [18]:
nb_model=joblib.load("nb_model.joblib")
svm_model=joblib.load("svm_model.joblib")

In [28]:
nb_y_pred=nb_model.predict(X_test)
nb_report=classification_report(y_test,nb_y_pred,output_dict=True)
print(nb_report)

{'ATC Issue': {'precision': 0.5707070707070707, 'recall': 0.8218181818181818, 'f1-score': 0.6736214605067065, 'support': 275.0}, 'Conflict': {'precision': 0.6730769230769231, 'recall': 0.5691056910569106, 'f1-score': 0.6167400881057269, 'support': 123.0}, 'Deviation': {'precision': 0.7605042016806722, 'recall': 0.6146010186757216, 'f1-score': 0.67981220657277, 'support': 589.0}, 'Equipment Problem': {'precision': 0.9002217294900222, 'recall': 0.9362029208301307, 'f1-score': 0.9178598342125094, 'support': 1301.0}, 'Ground Event': {'precision': 1.0, 'recall': 0.17142857142857143, 'f1-score': 0.2926829268292683, 'support': 35.0}, 'Inflight Event': {'precision': 0.5714285714285714, 'recall': 0.45901639344262296, 'f1-score': 0.509090909090909, 'support': 61.0}, 'accuracy': 0.8011744966442953, 'macro avg': {'precision': 0.74598974939721, 'recall': 0.5953621295420232, 'f1-score': 0.6149679042196482, 'support': 2384.0}, 'weighted avg': {'precision': 0.8090249554095988, 'recall': 0.801174496644

In [29]:
svm_y_pred=svm_model.predict(X_test)
svm_report=classification_report(y_test,svm_y_pred,output_dict=True)
print(svm_report)

{'ATC Issue': {'precision': 0.6970802919708029, 'recall': 0.6945454545454546, 'f1-score': 0.6958105646630237, 'support': 275.0}, 'Conflict': {'precision': 0.7171717171717171, 'recall': 0.5772357723577236, 'f1-score': 0.6396396396396397, 'support': 123.0}, 'Deviation': {'precision': 0.735, 'recall': 0.7487266553480475, 'f1-score': 0.7417998317914214, 'support': 589.0}, 'Equipment Problem': {'precision': 0.9019896831245394, 'recall': 0.940814757878555, 'f1-score': 0.9209932279909706, 'support': 1301.0}, 'Ground Event': {'precision': 0.75, 'recall': 0.17142857142857143, 'f1-score': 0.27906976744186046, 'support': 35.0}, 'Inflight Event': {'precision': 0.7391304347826086, 'recall': 0.5573770491803278, 'f1-score': 0.6355140186915887, 'support': 61.0}, 'accuracy': 0.8250838926174496, 'macro avg': {'precision': 0.7567286878416114, 'recall': 0.61502137678978, 'f1-score': 0.6521378417030842, 'support': 2384.0}, 'weighted avg': {'precision': 0.8211618019173057, 'recall': 0.8250838926174496, 'f1-

In [ ]:
runs=[
    {
        "run_name":"Naive Bayes",
        "model_type":"multinomial_naive_bayes",
        "params":{
            "max_features":10000,
            "ngram_range":"(1,1)",
            "min_df":1,
            "alpha":0.1,
            "stop_words":"english",
            "random_state":42,
        },
        "report":nb_report
    },
    {
        "run_name":"Support Vector Machine",
        "model_type":"linear_svm",
        "params":{
            "max_features":20000,
            "ngram_range":"(1,2)",
            "min_df":1,
            "C":1.0,
            "stop_words":"english",
            "random_state":42,
        },
        "report":svm_report
    }
]



In [31]:
import mlflow
mlflow.set_experiment(experiment_name="aviation-classification")
for model in runs:
    with mlflow.start_run(run_name=model["run_name"]):
     mlflow.log_param("model_type",model["model_type"])
     for para_key,para_values in model["params"].items():
        mlflow.log_param(para_key,para_values)
     report=model["report"] 
     mlflow.log_metric("test_accuracy",report["accuracy"])
     mlflow.log_metric("macro_f1",report["macro avg"]["f1-score"])
     mlflow.log_metric("weighted_f1",report["weighted avg"]["f1-score"])
     mlflow.log_metric("ground_event_f1",report["Ground Event"]["f1-score"])
     mlflow.log_metric("ground_event_recall",report["Ground Event"]["recall"])
     
